**COMMAND:** `systemctl` deep dive

**PURPOSE (support):** You already know `systemctl start/stop/status` at a basic level. In support work, you live in `systemctl` — it's how you find out **_why_ a service is down**, whether it's supposed to auto-start on boot, and whether someone deliberately disabled it. The subcommands beyond basic start/stop are what separate "I restarted it and hoped" from "I diagnosed exactly what state it was in and why."

**⚠️ WSL FLAG:** systemd is _not_ enabled by default in WSL. Ubuntu 22.04+ on WSL2 supports it, but only if `/etc/wsl.conf` has:

In [ ]:
dennie@HP:~/linux-tut$ cat /etc/wsl.conf | head -n 5
[boot]
systemd=true

Paste the output — if `systemd=true` isn't there, we'll need to add it and run `wsl --shutdown` from PowerShell before continuing (that restarts the whole WSL VM, not just this terminal).

**SYNTAX:** `systemctl [command] [unit]` — most commands need `sudo` since they change system state.

**FLAGS / SUBCOMMANDS:**

|Subcommand|Sudo?|Meaning|
|---|---|---|
|`status UNIT`|no|current state, recent log lines, PID, memory|
|`start UNIT`|yes|start now (not persistent across reboot)|
|`stop UNIT`|yes|stop now|
|`restart UNIT`|yes|full stop + start|
|`reload UNIT`|yes|re-read config **without** dropping connections (like SIGHUP) — not all services support this|
|`enable UNIT`|yes|start automatically on boot (symlinks into a target) — does **not** start it now|
|`disable UNIT`|yes|remove from boot startup — does **not** stop it now|
|`mask UNIT`|yes|hard-block — even manual `start` will refuse until unmasked. Stronger than disable.|
|`list-units --type=service`|no|all currently loaded services and their state|
|`list-unit-files`|no|all installed services and whether enabled/disabled/static|

**Developer bridge:** `enable`/`disable` is exactly like a Django app being in `INSTALLED_APPS` vs not — it controls whether it participates automatically, separate from whether it's currently running. `start`/`stop` is the runtime toggle, like running `runserver` right now regardless of config.

**EXAMPLE — cron, a real service in your curriculum, good to practice on:**

In [ ]:
$ systemctl status cron
● cron.service - Regular background program processing daemon
     Loaded: loaded (/lib/systemd/system/cron.service; enabled; vendor preset: enabled)
     Active: active (running) since Tue 2026-09-08 09:01:12 UTC; 2h 14min ago
   Main PID: 512 (cron)
      Tasks: 1 (limit: 4915)
     Memory: 1.2M
        CPU: 89ms
     CGroup: /system.slice/cron.service
             └─512 /usr/sbin/cron -f

**READ THE OUTPUT:**

- `Loaded: ... enabled` — will auto-start on boot
- `Active: active (running)` — currently up, with exact start time and uptime
- `Main PID` — feed this straight into `ps -p 512` or `kill` if needed
- The `●` dot is color-coded: green = active/good, red = failed, white = inactive/stopped

A **failed** service looks like this instead:

In [ ]:
● cron.service - Regular background program processing daemon
     Loaded: loaded (/lib/systemd/system/cron.service; enabled; vendor preset: enabled)
     Active: failed (Result: exit-code) since Tue 2026-09-08 11:20:03 UTC; 4s ago
    Process: 8821 ExecStart=/usr/sbin/cron -f (code=exited, status=1/FAILURE)

Red `●`, `Active: failed`, and an exit code — that exit code (`1` here) is your next lead, cross-referenced against `journalctl -u cron` (next lesson).

**TRY NOW:**

In [ ]:
dennie@HP:~/linux-tut$ sudo systemctl status cron
[sudo] password for dennie:
● cron.service - Regular background >
     Loaded: loaded (/usr/lib/system>
     Active: active (running) since >
       Docs: man:cron(8)
   Main PID: 159 (cron)
      Tasks: 1 (limit: 9427)
     Memory: 444.0K (peak: 2.1M)
        CPU: 101ms
     CGroup: /system.slice/cron.serv>
             └─159 /usr/sbin/cron -f>

Sep 08 03:48:46 HP systemd[1]: Start>
Sep 08 03:48:46 HP (cron)[159]: cron>
Sep 08 03:48:46 HP cron[159]: (CRON)>
Sep 08 03:48:46 HP cron[159]: (CRON)>
Sep 08 04:17:00 HP CRON[4964]: pam_u>
Sep 08 04:17:00 HP CRON[4965]: (root>
Sep 08 04:17:00 HP CRON[4964]: pam_u>
Sep 08 05:17:00 HP CRON[11169]: pam_>
Sep 08 05:17:00 HP CRON[11170]: (roo>
Sep 08 05:17:00 HP CRON[11169]: pam_>
lines 1-21/21 (END)

In [ ]:
dennie@HP:~/linux-tut$ sudo systemctl stop cron

In [ ]:
dennie@HP:~/linux-tut$ systemctl status cron
○ cron.service - Regular background >
     Loaded: loaded (/usr/lib/system>
     Active: inactive (dead) since T>
   Duration: 12h 21min 28.600s
       Docs: man:cron(8)
    Process: 159 ExecStart=/usr/sbin>
   Main PID: 159 (code=killed, signa>
        CPU: 102ms

Sep 08 03:48:46 HP cron[159]: (CRON)>
Sep 08 04:17:00 HP CRON[4964]: pam_u>
Sep 08 04:17:00 HP CRON[4965]: (root>
Sep 08 04:17:00 HP CRON[4964]: pam_u>
Sep 08 05:17:00 HP CRON[11169]: pam_>
Sep 08 05:17:00 HP CRON[11170]: (roo>
Sep 08 05:17:00 HP CRON[11169]: pam_>
Sep 08 16:10:15 HP systemd[1]: Stopp>
Sep 08 16:10:15 HP systemd[1]: cron.>
Sep 08 16:10:15 HP systemd[1]: Stopp>
lines 1-19/19 (END)

In [ ]:
dennie@HP:~/linux-tut$ sudo systemctl start cron
dennie@HP:~/linux-tut$ systemctl is-enabled cron
enabled

(`is-enabled` is a quick one-word check — prints `enabled`, `disabled`, `static`, or `masked` — handy in scripts instead of parsing full `status` output.)

Excellent — clean run all the way through, and systemd is properly enabled. A few notes on what you saw:

**The truncated lines** (`● cron.service - Regular background >`) — that's just your terminal window being narrow enough that `systemctl status` truncated with `>`, not a bug. Widen the VS Code terminal pane, or add `--no-pager` flag to prevent the interactive pager from also wrapping/reflowing awkwardly:

In [ ]:
systemctl status cron --no-pager

You correctly escaped it with `Ctrl+C`, but the idiomatic way out of a pager is `q` (quit) — `Ctrl+C` works but can occasionally leave the terminal in a slightly odd state (garbled colors/echo) since it's an interrupt signal, not a "quit the pager cleanly" command. Worth building the `q` habit since you'll be paging through `journalctl` and `less` constantly in support work.

**The stop/start cycle — read closely, this confirms the signal theory from earlier:**

In [ ]:
Main PID: 159 (code=killed, signal=TERM)

That's systemd itself telling you: it stopped cron by sending **SIGTERM** (signal 15) — exactly the graceful-first signal from the `kill` lesson, applied automatically under the hood. This is why `reload` exists as an alternative for services that support it — `stop`/`start` always tears down and rebuilds the process via SIGTERM, `reload` (where supported) just re-reads config in place, no PID change, no SIGTERM.

**Final state check — you're good:**

In [ ]:
Active: active (running)  # ← from your last implicit start
enabled                  # ← systemctl is-enabled cron

Correctly running and correctly set to persist across reboot.

---

**TICKET SCENARIO:** "Cron jobs stopped running overnight." Diagnosis path:

1. `systemctl status cron` — is it even running? If `inactive` or `failed`, that alone explains everything.
2. `systemctl is-enabled cron` — if `disabled`, it won't survive the next reboot even if you `start` it now — you'd need `sudo systemctl enable cron` too.
3. If `masked` — someone deliberately blocked it; `start` will refuse and you'll need `sudo systemctl unmask cron` first.

**GOTCHA:** `restart` vs `reload` — always check `systemctl status UNIT` first for `reload` support; not every service implements it. Using `restart` when `reload` was possible causes a real (if brief) outage/dropped connections you didn't need. Also: `stop` + reboot the box will NOT bring it back if it's `disabled` — `stop` and `disable` are independent axes, a very common WSL-specific gotcha too since WSL VM restarts (`wsl --shutdown`) are common during setup work.